<a href="https://colab.research.google.com/github/LeoCH25/Programaci-n-de-Sistemas-Base-1/blob/main/Unidad%203/MI%20Copia%20de%20analizador_lexico_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GUÍA PRÁCTICA Y EVALUACIÓN ASÍNCRONA: ANALIZADORES LÉXICOS EN PYTHON Y JAVA
**Tema: Generación Automática de Analizadores Léxicos**  
**Asignatura:** Programación de Sistemas de Bas 1  

---  
## 🎯 Objetivos de Aprendizaje
1. **Comprender** la generación automática de analizadores léxicos a partir de especificaciones formales y expresiones regulares.
2. **Implementar y extender** escáneres léxicos en dos entornos tecnológicos representativos: **PLY (`ply.lex`)** en Python y **JFlex** en Java.
3. **Analizar** los mecanismos de disambiguación léxica (palabras reservadas vs. identificadores, coincidencia más larga) y el manejo de atributos/posiciones.
4. **Evaluar** estructuralmente las diferencias entre enfoques interpretados/dinámicos (PLY) y compilados/estáticos (JFlex).

---  
## 🐍 PARTE 1: Analizador Léxico en Python utilizando PLY (`ply.lex`)

### 1.1 Código Base e Instalación

In [ ]:
# Instalación de la librería PLY en el entorno de Google Colab
!pip install ply > /dev/null

In [ ]:
%%writefile lexer_ply.py

import ply.lex as lex

class PythonLexerPLY:
    """Especificación e implementación de un Analizador Léxico en Python usando PLY."""

    # 1. LISTA OBLIGATORIA DE TOKENS
    tokens = (
        'KEYWORD_IF',
        'KEYWORD_ELSE',
        'KEYWORD_WHILE',
        'KEYWORD_INT',
        'KEYWORD_FLOAT',
        'IDENTIFIER',
        'INTEGER_LITERAL',
        'FLOAT_LITERAL',
        'ASSIGN',
        'PLUS',
        'MINUS',
        'TIMES',
        'DIVIDE',
        'SEMICOLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
    )

    # 2. DICCIONARIO ASOCIATIVO DE PALABRAS RESERVADAS
    reserved = {
        'if': 'KEYWORD_IF',
        'else': 'KEYWORD_ELSE',
        'while': 'KEYWORD_WHILE',
        'int': 'KEYWORD_INT',
        'float': 'KEYWORD_FLOAT'
    }

    # 3. REGLAS SIMPLES DE CADENA (t_TOKEN)
    t_ASSIGN    = r'='
    t_PLUS      = r'\+'
    t_MINUS     = r'-'
    t_TIMES     = r'\*'
    t_DIVIDE    = r'/'
    t_SEMICOLON = r';'
    t_LPAREN    = r'\('
    t_RPAREN    = r'\)'
    t_LBRACE    = r'\{'
    t_RBRACE    = r'\}'

    # Caracteres ignorados (espacios y tabuladores)
    t_ignore = ' \t'

    # 4. REGLAS BASADAS EN FUNCIONES (t_TOKEN(t))
    def t_FLOAT_LITERAL(self, t):
        r'\d+\.\d+([eE][+-]?\d+)?'
        t.value = float(t.value)
        return t

    def t_INTEGER_LITERAL(self, t):
        r'\d+'
        t.value = int(t.value)
        return t

    def t_IDENTIFIER(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        # Disambiguación entre palabras reservadas e identificadores de usuario
        t.type = self.reserved.get(t.value, 'IDENTIFIER')
        return t

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"⚠️  [Error Léxico] Línea {t.lexer.lineno}, Posición {t.lexpos}: Carácter ilegal '{t.value[0]}'")
        t.lexer.skip(1)

    def build(self, **kwargs):
        """Construye el objeto lexer dinámico mediante PLY."""
        self.lexer = lex.lex(object=self, **kwargs)

    def test(self, data: str):
        """Prueba el escáner sobre una cadena fuente imprimiendo los tokens capturados."""
        self.lexer.input(data)
        print(f"{'Línea':<8} | {'Tipo de Token':<18} | {'Lexema / Valor':<20} | {'Posición Abs.':<12}")
        print("-" * 65)
        for tok in self.lexer:
            print(f"{tok.lineno:<8} | {tok.type:<18} | {str(tok.value):<20} | {tok.lexpos:<12}")

# Práctica Base
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

#print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
#py_lexer = PythonLexerPLY()
#py_lexer.build()
#py_lexer.test(sample_code)

Overwriting lexer_ply.py


In [ ]:
from lexer_ply import PythonLexerPLY

# Definimos la variable sample_code que no estaba cargada en el entorno
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
py_lexer = PythonLexerPLY()
py_lexer.build()
py_lexer.test(sample_code)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
2        | KEYWORD_INT        | int                  | 1           
2        | IDENTIFIER         | contador             | 5           
2        | ASSIGN             | =                    | 14          
2        | INTEGER_LITERAL    | 10                   | 16          
2        | SEMICOLON          | ;                    | 18          
3        | KEYWORD_FLOAT      | float                | 20          
3        | IDENTIFIER         | promedio             | 26          
3        | ASSIGN             | =                    | 35          
3        | FLOAT_LITERAL      | 95.5                 | 37          
3        | SEMICOLON          | ;                    | 41          
4        | KEYWORD_WHILE      | while                | 43          
4        | LPAREN             | (                    |

### 🛠️ Desafío 1 (PLY - Python)
**Instrucciones:** Escribe a continuación la extensión de la clase `PythonLexerPLY` (o redefine la clase ampliada `PythonLexerPLYExtended`) para incluir:
1. Soporte para **cadenas literales** entrecomilladas (`"..."`) asignando el token `'STRING_LITERAL'`.
2. Soporte para **comentarios de una línea** (`//...`), configurando el lexer para que sean **ignorados en silencio** sin emitir ningún token.

In [ ]:
# [ESPACIO PARA EL ALUMNO] - Escribe tu código ampliado de PLY aquí
# Pista: Agrega 'STRING_LITERAL' a tokens y define t_STRING_LITERAL y t_ignore_COMMENT
%%writefile lexer_ply_ext.py

from lexer_ply import PythonLexerPLY

class PythonLexerPLYExtended(PythonLexerPLY):
    """Extensión del analizador léxico: cadenas literales y comentarios de una línea."""

    # 1. Se heredan los tokens de la clase base y se agrega el nuevo
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # 2. Cadenas entre comillas dobles: admite escapes (\" o \n), pero no saltos de línea reales
    t_STRING_LITERAL = r'"(?:[^"\\\n]|\\.)*"'

    # 3. Comentarios de una línea: el prefijo t_ignore_ hace que PLY los descarte sin emitir token
    t_ignore_COMMENT = r'//.*'

Overwriting lexer_ply_ext.py


In [ ]:
from lexer_ply_ext import PythonLexerPLYExtended

sample_code_ext = """
// Este comentario no debe generar tokens
int contador = 10; // comentario al final de la línea
float promedio = 95.5 / 2.0;
mensaje = "Hola, mundo";
ruta = "http://sitio.com"; // las // dentro de una cadena no son comentario
cita = "Dijo \\"hola\\"";
"""

print("=== PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO (PLY) ===")
ext_lexer = PythonLexerPLYExtended()
ext_lexer.build()
ext_lexer.test(sample_code_ext)

=== PRUEBA DEL ANALIZADOR LÉXICO EXTENDIDO (PLY) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
3        | KEYWORD_INT        | int                  | 43          
3        | IDENTIFIER         | contador             | 47          
3        | ASSIGN             | =                    | 56          
3        | INTEGER_LITERAL    | 10                   | 58          
3        | SEMICOLON          | ;                    | 60          
4        | KEYWORD_FLOAT      | float                | 97          
4        | IDENTIFIER         | promedio             | 103         
4        | ASSIGN             | =                    | 112         
4        | FLOAT_LITERAL      | 95.5                 | 114         
4        | DIVIDE             | /                    | 119         
4        | FLOAT_LITERAL      | 2.0                  | 121         
4        | SEMICOLON          | ;                    | 124      

---  
## ☕ PARTE 2: Analizador Léxico en Java utilizando JFlex

### 2.1 Configuración e Instalación del Entorno

In [ ]:
import os

print("=== 1. INSTALANDO JDK EN GOOGLE COLAB ===")
ret = os.system("apt-get update -qq && apt-get install -y openjdk-11-jdk-headless > /dev/null")
print("✅ Entorno Java 11 configurado correctamente.\n" if ret == 0 else "❌ Falló la instalación del JDK.\n")

=== 1. INSTALANDO JDK EN GOOGLE COLAB ===
✅ Entorno Java 11 configurado correctamente.



### 2.2 Creación de Archivos Base de Java y Especificación JFlex

In [ ]:
# --- PASO 1: DEFINICIÓN DE LA ENUMERACIÓN DE TOKENS (TokenType.java) ---
tokentype_code = """
public enum TokenType {
    // Palabras Reservadas
    KEYWORD_IF, KEYWORD_ELSE, KEYWORD_WHILE, KEYWORD_INT, KEYWORD_FLOAT,

    // Identificadores y Literales
    IDENTIFIER, INTEGER_LITERAL, FLOAT_LITERAL, STRING_LITERAL,

    // Operadores y Delimitadores
    ASSIGN, PLUS, MINUS, TIMES, DIVIDE, SEMICOLON,
    LPAREN, RPAREN, LBRACE, RBRACE,

    // Control de Errores
    ERROR
}
"""

with open("TokenType.java", "w") as f:
    f.write(tokentype_code)

# --- PASO 2: CLASE CONTENEDORA DE TOKENS CON RASTREO (Token.java) ---
token_code = """
public class Token {
    public TokenType type;
    public Object value;
    public int line;
    public int column;

    public Token(TokenType type, Object value, int line, int column) {
        this.type = type;
        this.value = value;
        this.line = line;
        this.column = column;
    }

    @Override
    public String toString() {
        return String.format("%-8d | %-8d | %-18s | %-20s", line, column, type, value);
    }
}
"""

with open("Token.java", "w") as f:
    f.write(token_code)

# --- PASO 3: ARCHIVO DE ESPECIFICACIÓN JFLEX (scanner.jflex) ---
jflex_spec_code = """
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \\t\\r\\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }
"""

with open("scanner.jflex", "w") as f:
    f.write(jflex_spec_code)

# --- PASO 4: CLASE PRINCIPAL DE PRUEBA (Main.java) ---
main_code = """
import java.io.*;

public class Main {
    public static void main(String[] args) {
        if (args.length == 0) {
            System.out.println("Por favor ingrese la ruta del archivo de prueba.");
            return;
        }

        try {
            FileReader reader = new FileReader(args[0]);
            Lexer scanner = new Lexer(reader);

            System.out.println("=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===");
            System.out.println(String.format("%-8s | %-8s | %-18s | %-20s", "Línea", "Columna", "Tipo de Token", "Lexema / Valor"));
            System.out.println("-".repeat(65));

            Token token;
            while ((token = scanner.yylex()) != null) {
                System.out.println(token);
            }
            System.out.println("\\n=== ESCANEO FINALIZADO EXITOSAMENTE ===");

        } catch (IOException e) {
            System.err.println("Error al leer el archivo: " + e.getMessage());
        }
    }
}
"""

with open("Main.java", "w") as f:
    f.write(main_code)

# --- PASO 5: ARCHIVO DE FUENTE DE PRUEBA (prueba.txt) ---
sample_code = """int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba.txt", "w") as f:
    f.write(sample_code)

print("✅ Archivos Java y especificacion scanner.jflex creados exitosamente.")

✅ Archivos Java y especificacion scanner.jflex creados exitosamente.


### 2.3 Compilación y Ejecución en Colab

In [ ]:
print("=== DESCARGANDO JFLEX (JAR con dependencias) ===")
!rm -rf jflex-1.9.1 jflex.tar.gz
!wget -q https://github.com/jflex-de/jflex/releases/download/v1.9.1/jflex-1.9.1.tar.gz -O jflex.tar.gz
!tar -xzf jflex.tar.gz

print("\n=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===")
!java -Xlog:disable Main prueba.txt

=== DESCARGANDO JFLEX (JAR con dependencias) ===

=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===
Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | KEYWORD_INT        | int                 
1        | 5        | IDENTIFIER         | contador            
1        | 14       | ASSIGN             | =                   
1        | 16       | INTEGER_LITERAL    | 10                  
1        | 18       | SEMICOLON          | ;                   
2        | 1        | KEYWORD_FLOAT      | float               
2        | 7        | IDENTIFIER         | promedio            


### 🛠️ Desafío 2 (JFlex - Java)
**Instrucciones:** Modifica la especificación `scanner.jflex` arriba o vuelve a escribir el archivo desde una celda agregando:
1. La definición de la regla de **cadenas literales entrecomilladas**: `\"[^\"]*\" { return symbol(TokenType.STRING_LITERAL, yytext()); }`
2. La regla para **comentarios de una línea**: `"//".* { /* Ignorar */ }`
Recompila con `!jflex scanner.jflex && javac *.java` y ejecuta sobre un archivo de prueba ampliado.

In [ ]:
prueba_ampliada = """// Archivo de prueba ampliado: este comentario no genera tokens
int contador = 10;
float promedio = 95.5 / 2.0; // comentario al final de la línea
mensaje = "Hola, mundo";
ruta = "http://sitio.com"; // las // dentro de una cadena no son comentario
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba_ampliada.txt", "w") as f:
    f.write(prueba_ampliada)

!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java
!java -Xlog:disable Main prueba_ampliada.txt

Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Old file "Lexer.java" saved as "Lexer.java~"
Writing code to "Lexer.java"
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | DIVIDE             | /                   
1        | 2        | DIVIDE             | /                   
1        | 4        | IDENTIFIER         | Archivo             
1        | 12       | IDENTIFIER         | de                  
1        | 15       | IDENTIFIER         | prueba              
1        | 22       | IDENTIFIER         | ampliado            
⚠️  [Error Léxico] Línea 1, Columna 30: Carácter ilegal ':'
1        | 30       | ERROR              | :                   
1        | 32       | IDENTIFIER         | este     

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** _____________________________________  
**Matrícula / Código:** _______________________________________  
**Fecha:** _____________________  

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.  
   *Respuesta del alumno:*
   >

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?  
   *Respuesta del alumno:*
   >

---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   *Respuesta del alumno:*
   >

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.  
   *Respuesta del alumno:*
   >

---
### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
# Pegar aquí el fragmento de código
```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* Pegar aquí las reglas de JFlex */
```

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | (Interpretado / Reflexivo / Compilado) | (Interpretado / Reflexivo / Compilado) |
| **Definición de Patrones** | Docstrings en funciones Python | Archivo de especificación `.jflex` |
| **Manejo de Palabras Reservadas** | | |
| **Rastreo de Línea y Columna** | | |
| **Recuperación de Errores** | | |

---  
**Conclusión General del Experimento:**  
> *(Redacta una breve conclusión sobre las ventajas y desventajas de cada herramienta en el desarrollo de un compilador real)*

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** Leo Chavarria Uribe  
**Matrícula / Código:** 2223330145
**Fecha:** 12 de noviembre de 2026  

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.  
   *Respuesta del alumno:*
   > En **JFlex** el escáner es un AFD que no se detiene en el primer estado de aceptación: sigue leyendo mientras exista transición y recuerda el último estado de aceptación alcanzado. Con `1.2e2`:
   > 1. Lee `1` → estado de aceptación de `ENTERO` (se guarda como candidato), pero continúa.
   > 2. Lee `.` → estado intermedio de `FLOTANTE` (no acepta todavía).
   > 3. Lee `2` → estado de aceptación de `FLOTANTE` (`1.2`, nuevo candidato).
   > 4. Lee `e` → estado intermedio del exponente (no acepta).
   > 5. Lee `2` → estado de aceptación de `FLOTANTE` (`1.2e2`).
   > 6. El siguiente carácter (`;`) no tiene transición → el autómata se detiene y devuelve la coincidencia más larga: `FLOAT_LITERAL` con valor `120.0`, como se observa en la salida (línea 7, columna 27).
   >
   > Si después del punto no hubiera dígitos (por ejemplo `1.;`), el autómata retrocedería al último estado de aceptación y devolvería solo `ENTERO 1`. Además, `.` no es un token de este lenguaje y `2e2` no es un identificador válido (empieza con dígito), así que esa separación ni siquiera sería válida.
   >
   > En **PLY** el mecanismo es distinto: no construye un AFD, sino una expresión regular maestra con alternativas que el motor `re` de Python prueba **en orden** (gana la primera que coincide, no la más larga). Las reglas definidas como funciones se agregan en el orden en que aparecen en el código; como `t_FLOAT_LITERAL` está definida antes que `t_INTEGER_LITERAL`, se prueba primero y su patrón `\d+\.\d+([eE][+-]?\d+)?` consume `1.2e2` completo. Si el orden se invirtiera, PLY devolvería `1` como entero y marcaría `.` como error. En PLY la "regla más larga" depende del orden que define el programador.

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?  
   *Respuesta del alumno:*
   > **JFlex:** con la entrada `if`, tanto la regla `"if"` como `{ID}` reconocen exactamente 2 caracteres. Cuando dos reglas empatan en longitud, JFlex aplica la que aparece **primero** en la especificación, por lo que colocar las palabras reservadas arriba de `{ID}` hace que ganen el empate. El orden solo decide en caso de empate: con `ifx` o `whileTrue`, `{ID}` reconoce más caracteres y gana por la regla de la coincidencia más larga, así que esos lexemas se clasifican correctamente como `IDENTIFIER`.
   >
   > **PLY:** hay dos problemas con `t_KEYWORD_IF = r'if'`. (1) PLY agrega primero las reglas definidas como funciones y después las definidas como cadenas, así que `t_IDENTIFIER` siempre se probaría antes y `if` nunca llegaría a la regla de palabra reservada. (2) Aunque se probara primero, PLY usa "primera coincidencia" y no "más larga", así que un identificador como `ifcount` se partiría en `KEYWORD_IF` + `IDENTIFIER("count")`. Con `reserved.get()`, primero se reconoce la palabra completa con el patrón de identificador y luego se consulta el diccionario: si está ahí, se le cambia el tipo; si no, queda como `IDENTIFIER`. Además es más eficiente (búsqueda O(1) en un diccionario en lugar de una alternativa más en la expresión maestra) y más fácil de mantener: agregar una palabra reservada es agregar una entrada al diccionario.

---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   *Respuesta del alumno:*
   > Porque el analizador léxico es la única fase que trabaja con caracteres: su salida debe ser el par **⟨token, atributo⟩** con el atributo ya en su forma útil. Hacer la conversión ahí tiene varias ventajas:
   > * **Separación de responsabilidades:** el analizador sintáctico solo debe ver tipos de token (`FLOAT_LITERAL`), y el semántico necesita el valor numérico para verificar tipos, plegar constantes (`2 + 3 → 5`) y generar código. Ninguno debería volver a interpretar texto.
   > * **Normalización:** representaciones distintas del mismo valor quedan unificadas. En la práctica, `1.2e2` se entregó como `120.0`, igual que si se hubiera escrito `120.0`.
   > * **Detección temprana de errores:** un literal que no cabe en su tipo se detecta en el momento en que se lee, cuando todavía se conoce su línea y columna exactas. Por ejemplo, en la especificación de JFlex, `Integer.parseInt` lanzaría `NumberFormatException` con `99999999999`; lo correcto sería capturarla en la regla y reportarla como error léxico con su posición.
   > * **Eficiencia:** la conversión se hace una sola vez por token, en lugar de repetirse en cada fase que use el valor.

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.  
   *Respuesta del alumno:*
   > **PLY** solo rastrea automáticamente `t.lexpos`, que es la **posición absoluta** (índice desde 0) del token dentro de toda la cadena de entrada. El número de línea `t.lexer.lineno` (que empieza en 1) **no** se actualiza solo: PLY no sabe qué es un salto de línea, por eso el programador debe definir la regla `t_newline` que hace `t.lexer.lineno += len(t.value)`. La columna no existe; hay que calcularla a mano a partir de `lexpos`, buscando el último `\n` anterior (`columna = lexpos - data.rfind('\n', 0, lexpos)`). Tampoco se reinicia `lineno` al llamar otra vez a `input()`. En la práctica se ve el efecto: como `sample_code` empieza con un salto de línea, `int` aparece en la línea 2 con posición absoluta 1.
   >
   > **JFlex**, con las directivas `%line` y `%column`, genera código que actualiza automáticamente `yyline` y `yycolumn` mientras consume caracteres, reconociendo por sí mismo los terminadores de línea (`\n`, `\r\n`, `\r` y los de Unicode). Ambas variables empiezan en **0**, por eso el método `symbol()` les suma 1 antes de crear el `Token`. Si se necesitara la posición absoluta, existe la directiva `%char` con la variable `yychar`.
   >
   > En resumen: en PLY el rastreo es **manual** (línea) o **absoluto** (posición), mientras que en JFlex es **automático** y ya viene en coordenadas línea/columna, listo para los mensajes de error.

---
### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
class PythonLexerPLYExtended(PythonLexerPLY):
    # Se heredan los tokens de la clase base y se agrega el nuevo
    tokens = PythonLexerPLY.tokens + ('STRING_LITERAL',)

    # Cadenas entre comillas dobles: admite escapes (\" o \n), pero no saltos de línea reales
    t_STRING_LITERAL = r'"(?:[^"\\\n]|\\.)*"'

    # Comentarios de una línea: el prefijo t_ignore_ hace que PLY los descarte sin emitir token
    t_ignore_COMMENT = r'//.*'
```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* DESAFÍO 2.1: Cadenas literales entrecomilladas */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }

/* DESAFÍO 2.2: Comentarios de una línea (se descartan sin emitir token) */
"//".*          { /* Ignorar */ }
```

> **Resultado:** en ambas herramientas los comentarios dejaron de generar tokens, `"Hola, mundo"` y `"http://sitio.com"` se reconocieron como `STRING_LITERAL` (las `//` dentro de la cadena no se tomaron como comentario porque la regla de cadena comienza en la comilla y consume todo hasta la comilla de cierre), y la `/` sola siguió reconociéndose como `DIVIDE`, ya que `//.*` es una coincidencia más larga y tiene prioridad.

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | **Reflexivo / interpretado:** en tiempo de ejecución inspecciona la clase (`dir()`, docstrings) y arma una expresión regular maestra que ejecuta el motor `re` de Python. | **Compilado / estático:** antes de compilar traduce la especificación a código Java con un AFD minimizado (en la práctica: 47 estados → 42 tras la minimización). |
| **Definición de Patrones** | Docstrings en funciones Python y variables `t_NOMBRE` | Archivo de especificación `.jflex` (opciones, macros y reglas) |
| **Manejo de Palabras Reservadas** | Diccionario `reserved` consultado dentro de `t_IDENTIFIER`; una sola regla reconoce identificadores y palabras reservadas. | Una regla explícita por palabra reservada colocada antes de `{ID}`; el empate en longitud se resuelve por orden de aparición. |
| **Rastreo de Línea y Columna** | `lineno` se actualiza manualmente con `t_newline`; `lexpos` es posición absoluta automática; la columna se calcula a mano. | `%line` / `%column` actualizan `yyline` y `yycolumn` automáticamente (base 0); `%char` da la posición absoluta. |
| **Recuperación de Errores** | Función `t_error` con `t.lexer.skip(1)`: reporta, salta un carácter (modo pánico) y no emite token. | Regla comodín `.` al final: reporta con línea/columna y devuelve un token `ERROR`, de modo que el analizador sintáctico se entera del error. |

---  
**Conclusión General del Experimento:**  
> Ambas herramientas generan un analizador léxico a partir de expresiones regulares y produjeron resultados equivalentes: reconocieron los mismos tokens, convirtieron `1.2e2` a `120.0`, extendieron el lenguaje con cadenas y comentarios, y reportaron el carácter ilegal `@` sin detener el análisis.
>
> **PLY** destaca por su rapidez de desarrollo: no requiere un paso de compilación, se integra directamente con Python y se puede extender por herencia (el Desafío 1 se resolvió agregando tres líneas). Sus desventajas son que la prioridad de las reglas depende del orden de definición ("primera coincidencia", no "más larga"), lo que puede producir errores sutiles; que el rastreo de líneas y columnas es manual; que es más lento por usar un motor de expresiones regulares con retroceso; y que depende de introspección (en la práctica, la clase extendida tuvo que escribirse en un archivo `.py` porque PLY no la acepta definida directamente en una celda del cuaderno).
>
> **JFlex** genera un AFD minimizado que analiza en tiempo lineal y aplica de forma garantizada la regla de la coincidencia más larga; además, rastrea posiciones automáticamente y se integra con generadores de analizadores sintácticos como CUP. Sus desventajas son el ciclo más largo (generar → compilar → ejecutar), una sintaxis más verbosa y que el código generado es difícil de depurar; en la práctica también hubo que cuidar el escape de caracteres (`\"`) al escribir la especificación desde un string de Python.
>
> Para un compilador real, JFlex es la opción más robusta por su rendimiento y su comportamiento determinista; PLY es más adecuado para prototipos, lenguajes pequeños o herramientas educativas donde importa más la rapidez de desarrollo que el rendimiento.